# Ringkasan Pengerjaan

Augmentasi dataset lewat pemecahan kontrak jadi subgraf per fungsi, mengatasi kesenjangan ukuran graf antara data latih dan benchmark. Ini jadi **model utama penelitian**.

**Isi**: pelabelan ulang reentrancy di level fungsi memakai tiga detektor Slither sekaligus (ReentrancyEth, ReentrancyNoGas, ReentrancyBenign), ekstraksi CFG dengan atribut fungsi, pemecahan jadi subgraf, sampling dan pruning, digabung dengan data kontrak utuh jadi `v3_augmented.pt`. Dilanjutkan training model utama, evaluasi benchmark, kalibrasi ambang keputusan, analisis SHAP, dan ablation study (kedalaman arsitektur dan metode pooling).

**Hasil akhir**: 1.712 graf latih (1.014 aman, 698 rentan). F1 cross-validation 87.98%, AUC cross-validation 94.91%. AUC benchmark naik ke 70.31% (dari 56 sampai 59% pada semua varian data v2), 6 dari 8 pasangan CEI benar. Rata-rata IoU SHAP 0.150, turun dari 0.277 di model v2 (model lebih aktif membaca struktur, namun kurang tepat sasaran). Ablation menunjukkan metode pooling lebih menentukan performa dibanding kedalaman arsitektur.

**Catatan**: dataset dan model di notebook ini adalah hasil akhir yang dipakai untuk penulisan Bab 4, dibangun di atas `PROJECT_v2.ipynb`.

# Pelabelan Reentrancy per Fungsi dan Ekstraksi Subgraf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q slither-analyzer solc-select torch_geometric

import os, re, json, time, subprocess
import numpy as np, pandas as pd
from packaging.specifiers import SpecifierSet
from packaging.version import Version
from slither import Slither
from solc_select import solc_select
import logging
logging.getLogger('CryticCompile').setLevel(logging.ERROR)

D = '/content/drive/MyDrive/PROJECT'
BASE = '/content/data/raw/dataset_v2'
MANIFEST = f'{D}/data/processed/v2_manifest.csv'

# kembalikan binary solc dari Drive
if not os.path.exists(os.path.expanduser('~/.solc-select/artifacts')):
    os.makedirs(os.path.expanduser('~/.solc-select'), exist_ok=True)
    !cp -r {D}/solc_binaries/artifacts ~/.solc-select/
    !chmod -R +x ~/.solc-select/artifacts/

# kembalikan file .sol dari zip v2
if not os.path.exists(f'{BASE}/1_reentrancy') or len(os.listdir(f'{BASE}/1_reentrancy')) == 0:
    os.makedirs('/content/data/raw', exist_ok=True)
    !unzip -q -o {D}/data/raw/dataset_v2.zip -d /content/data/raw/

terpasang = sorted([Version(v) for v in solc_select.installed_versions()])
print(f'solc terpasang: {len(terpasang)}')
print(f'normal   : {len(os.listdir(f"{BASE}/0_normal"))}')
print(f'reentrancy: {len(os.listdir(f"{BASE}/1_reentrancy"))}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
solc terpasang: 61
normal   : 500
reentrancy: 244


In [ ]:
import shutil
shutil.rmtree(os.path.expanduser('~/.solc-select'), ignore_errors=True)
os.makedirs(os.path.expanduser('~/.solc-select'), exist_ok=True)
!cp -r {D}/solc_binaries/artifacts ~/.solc-select/
!chmod -R +x ~/.solc-select/artifacts/

terpasang = sorted([Version(v) for v in solc_select.installed_versions()])
print(f'solc terpasang: {len(terpasang)}')

solc terpasang: 61


In [ ]:
print(f"ukuran file di Drive: {os.path.getsize(f'{D}/data/raw/dataset_v2.zip')} bytes")
!unzip -t {D}/data/raw/dataset_v2.zip 2>&1 | tail -20

ukuran file di Drive: 11264431 bytes
    testing: dataset_v2/1_reentrancy/0x1c839fd542563182BFD0C337c8AaD1Be412F0fd2.sol   OK
    testing: dataset_v2/1_reentrancy/0x00ef6d929b76dfbf4b87d3796c76920708314fd1.sol   OK
    testing: dataset_v2/1_reentrancy/0xe59b261d3ea188c952ff254ffaf270568ae3ef02.sol   OK
    testing: dataset_v2/1_reentrancy/0x5f906a6af3f5d52dc84e0e472e95501f24ebaef0.sol   OK
    testing: dataset_v2/1_reentrancy/0x2A86c7a265FD2497bC8aE5ceC8E1b0cb19359FD5.sol   OK
    testing: dataset_v2/1_reentrancy/0x2Bf38de07042240b52B7C40F044ABEfD7b01A5BE.sol   OK
    testing: dataset_v2/1_reentrancy/0x00edca4652f6e117c0c1f04f2280a86ae08bdd0c.sol   OK
    testing: dataset_v2/1_reentrancy/0x00ce2071edbe54709ae8a4c1d4e0fc4e5d2ae6ef.sol   OK
    testing: dataset_v2/1_reentrancy/0xaeda53544201731426b6e07cfe3b396e0a1d40dd.sol   OK
    testing: dataset_v2/1_reentrancy/0x4b16df6737101d43df0db6e5306f5524fbe53a63.sol   OK
    testing: dataset_v2/1_reentrancy/0xbbdE1047EE332A3798AD9028714217622d

In [ ]:
os.makedirs('/content/data/raw', exist_ok=True)
!unzip -q -o {D}/data/raw/dataset_v2.zip -d /content/data/raw/

print(f'normal    : {len(os.listdir(f"{BASE}/0_normal"))}')
print(f'reentrancy: {len(os.listdir(f"{BASE}/1_reentrancy"))}')

normal    : 500
reentrancy: 244


In [ ]:
def get_pragma(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        c = f.read()
    m = re.search(r'pragma\s+solidity\s*([^;]+);', c)
    return m.group(1).strip() if m else 'TIDAK_DITEMUKAN'

def resolve_solc_v2(pragma_str):
    if not pragma_str or str(pragma_str) == 'TIDAK_DITEMUKAN':
        return None
    s = str(pragma_str).replace('^', '~=').strip()
    s = re.sub(r'([<>=~!]+)\s+', r'\1', s)
    try:
        spec = SpecifierSet(','.join(p.strip() for p in s.split() if p.strip()))
        cocok = [v for v in terpasang if v in spec]
        if cocok:
            return str(max(cocok))
    except Exception:
        pass
    m = re.search(r'(\d+\.\d+\.\d+)', str(pragma_str))
    return m.group(1) if m else None

def resolve_solc_multi(fp):
    with open(fp, encoding='utf-8', errors='replace') as f:
        isi = f.read()
    semua = re.findall(r'pragma\s+solidity\s*([^;]+);', isi)
    if not semua:
        return None
    kandidat = [resolve_solc_v2(p.strip()) for p in semua]
    kandidat = [Version(v) for v in kandidat if v]
    return str(max(kandidat)) if kandidat else None

print('fungsi resolver siap')

fungsi resolver siap


In [ ]:
from slither.detectors.reentrancy.reentrancy_eth import ReentrancyEth

def label_fungsi_reentrancy(fp, versi_solc):
    """Kembalikan set nama fungsi yang terdeteksi reentrancy-eth di file ini."""
    subprocess.run(['solc-select', 'use', versi_solc], capture_output=True)
    sl = Slither(fp)
    sl.register_detector(ReentrancyEth)
    rentan = set()
    try:
        hasil_semua = sl.run_detectors()
    except Exception as e:
        print('  (run_detectors gagal:', str(e)[:100], ')')
        return rentan
    for hasil_detektor in hasil_semua:
        for h in hasil_detektor:
            for elem in h.get('elements', []):
                if elem.get('type') == 'function':
                    rentan.add(elem['name'])
    return rentan

print('fungsi label siap (pakai register_detector + run_detectors)')

fungsi label siap (pakai register_detector + run_detectors)


In [ ]:
ver = resolve_solc_multi(fp_sol)
hasil = label_fungsi_reentrancy(fp_sol, ver)
print(f'versi: {ver}')
print(f'fungsi terdeteksi rentan: {hasil}')

versi: 0.4.26
fungsi terdeteksi rentan: {'withdraw'}


In [ ]:
mf1 = pd.read_csv(f'{D}/data/processed/manifest_final.csv')
print('kolom:', list(mf1.columns))
print('isi unik folder:', mf1.folder.unique())
df_ren = mf1[(mf1.folder == '1_reentrancy') & (mf1.status == 'OK')].copy()
print('jumlah reentrancy OK:', len(df_ren))
print(df_ren[['nama_file','path','solc']].head(3))

kolom: ['path', 'nama_file', 'folder', 'label', 'split', 'pragma', 'solc', 'status', 'alasan', 'jumlah_fungsi', 'jumlah_node']
isi unik folder: ['0_normal' '1_reentrancy' '3_expert_benchmark/aman'
 '3_expert_benchmark/rentan']
jumlah reentrancy OK: 244
                                         nama_file  \
1   0x0000000000b3F879cb30FE243b4Dfee438691c04.sol   
6   0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol   
28  0x0014966c7439d8c10aaf0204036198236d9b0912.sol   

                                                 path    solc  
1   /content/data/raw/dataset_penelitian/1_reentra...  0.4.10  
6   /content/data/raw/dataset_penelitian/1_reentra...  0.4.11  
28  /content/data/raw/dataset_penelitian/1_reentra...  0.4.26  


In [ ]:
print('path pertama ada:', os.path.exists(df_ren.iloc[0].path))
if not os.path.exists(df_ren.iloc[0].path):
    print('perlu unzip dataset_final.zip dulu')

path pertama ada: True


In [ ]:
os.makedirs('/content/data/raw', exist_ok=True)
!unzip -q -n {D}/data/raw/dataset_final.zip -d /content/data/raw/
print('path pertama ada sekarang:', os.path.exists(df_ren.iloc[0].path))

path pertama ada sekarang: True


In [ ]:
print(f'{len(df_ren)} kontrak reentrancy akan dilabel ulang per fungsi')

hasil_label = {}
versi_aktif, mulai = None, time.time()
for i, (_, r) in enumerate(df_ren.iterrows(), 1):
    try:
        if r.solc != versi_aktif:
            subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
            versi_aktif = r.solc
        fungsi_rentan = label_fungsi_reentrancy(r.path, str(r.solc))
        hasil_label[r.nama_file] = fungsi_rentan
    except Exception as e:
        hasil_label[r.nama_file] = set()
    if i % 25 == 0:
        total_rentan = sum(len(v) > 0 for v in hasil_label.values())
        print(f'[{i}/{len(df_ren)}] {total_rentan} kontrak punya >=1 fungsi rentan ({(time.time()-mulai)/60:.1f} mnt)')

total_rentan = sum(len(v) > 0 for v in hasil_label.values())
total_fungsi_rentan = sum(len(v) for v in hasil_label.values())
print(f'\nselesai dalam {(time.time()-mulai)/60:.1f} menit')
print(f'{total_rentan}/{len(df_ren)} kontrak punya minimal 1 fungsi rentan terdeteksi')
print(f'{total_fungsi_rentan} total fungsi rentan terdeteksi')

import pickle
with open(f'{D}/data/processed/label_fungsi_reentrancy.pkl', 'wb') as f:
    pickle.dump(hasil_label, f)
print('tersimpan ke label_fungsi_reentrancy.pkl')

244 kontrak reentrancy akan dilabel ulang per fungsi
[25/244] 5 kontrak punya >=1 fungsi rentan (0.7 mnt)
[50/244] 5 kontrak punya >=1 fungsi rentan (5.5 mnt)
[75/244] 6 kontrak punya >=1 fungsi rentan (8.4 mnt)
[100/244] 20 kontrak punya >=1 fungsi rentan (9.1 mnt)
[125/244] 28 kontrak punya >=1 fungsi rentan (9.7 mnt)
[150/244] 38 kontrak punya >=1 fungsi rentan (10.5 mnt)
[175/244] 61 kontrak punya >=1 fungsi rentan (11.1 mnt)
  (run_detectors gagal: maximum recursion depth exceeded )
[200/244] 73 kontrak punya >=1 fungsi rentan (11.7 mnt)
[225/244] 91 kontrak punya >=1 fungsi rentan (12.3 mnt)

selesai dalam 12.7 menit
105/244 kontrak punya minimal 1 fungsi rentan terdeteksi
167 total fungsi rentan terdeteksi
tersimpan ke label_fungsi_reentrancy.pkl


In [ ]:
with open(f'{D}/data/processed/label_fungsi_reentrancy.pkl', 'rb') as f:
    hasil_label = pickle.load(f)

kosong = [nama for nama, fn in hasil_label.items() if len(fn) == 0]
print(f'{len(kosong)} kontrak dengan 0 fungsi rentan terdeteksi')

# cek 5 contoh secara manual, dengan pesan error yang sebenarnya (bukan try-except diam)
for nama in kosong[:5]:
    r = df_ren[df_ren.nama_file == nama].iloc[0]
    print(f'\n=== {nama} (solc {r.solc}) ===')
    try:
        subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
        sl = Slither(r.path)
        sl.register_detector(ReentrancyEth)
        hasil = sl.run_detectors()
        n_temuan = sum(len(h) for h in hasil)
        print(f'  Slither jalan normal, {n_temuan} temuan reentrancy-eth')
    except Exception as e:
        print(f'  ERROR: {str(e)[:150]}')

139 kontrak dengan 0 fungsi rentan terdeteksi

=== 0x0000000000b3F879cb30FE243b4Dfee438691c04.sol (solc 0.4.10) ===
  Slither jalan normal, 0 temuan reentrancy-eth

=== 0x0046f822ef4e6d51a40dd87d34ab1d5f088291a4.sol (solc 0.4.11) ===
  Slither jalan normal, 0 temuan reentrancy-eth

=== 0x0014966c7439d8c10aaf0204036198236d9b0912.sol (solc 0.4.26) ===
  Slither jalan normal, 0 temuan reentrancy-eth

=== 0x0062179227c03efc81301ade144f447f2561edd5.sol (solc 0.4.18) ===
  Slither jalan normal, 0 temuan reentrancy-eth

=== 0x00603fd28964766e9351c0827e2938cf4fb20d8d.sol (solc 0.4.19) ===
  Slither jalan normal, 0 temuan reentrancy-eth


In [ ]:
from slither.detectors.reentrancy.reentrancy_eth import ReentrancyEth
from slither.detectors.reentrancy.reentrancy_no_gas import ReentrancyNoGas
from slither.detectors.reentrancy.reentrancy_benign import ReentrancyBenign

DETEKTOR_SET = [ReentrancyEth, ReentrancyNoGas, ReentrancyBenign]

def label_fungsi_reentrancy_v2(fp, versi_solc):
    """Kembalikan set nama fungsi yang terdeteksi oleh salah satu dari 3 detektor reentrancy."""
    subprocess.run(['solc-select', 'use', versi_solc], capture_output=True)
    sl = Slither(fp)
    for Det in DETEKTOR_SET:
        sl.register_detector(Det)
    rentan = set()
    try:
        hasil_semua = sl.run_detectors()
    except Exception:
        return rentan
    for hasil_detektor in hasil_semua:
        for h in hasil_detektor:
            for elem in h.get('elements', []):
                if elem.get('type') == 'function':
                    rentan.add(elem['name'])
    return rentan

# uji cepat pada 5 kontrak yang tadi kosong
for nama in kosong[:5]:
    r = df_ren[df_ren.nama_file == nama].iloc[0]
    try:
        hasil = label_fungsi_reentrancy_v2(r.path, str(r.solc))
        print(f'{nama[:40]:42} -> {hasil if hasil else "masih kosong"}')
    except Exception as e:
        print(f'{nama[:40]:42} -> ERROR: {str(e)[:80]}')

0x0000000000b3F879cb30FE243b4Dfee438691c   -> masih kosong
0x0046f822ef4e6d51a40dd87d34ab1d5f088291   -> {'oraclize_newRandomDSQuery', 'makeOraclizeQuery', 'collect', 'oraclize_query', 'determineWinner', '__callback', 'payOwners', 'emptyRemainingsToOwners'}
0x0014966c7439d8c10aaf0204036198236d9b09   -> {'releaseTokensTo'}
0x0062179227c03efc81301ade144f447f2561ed   -> {'refund'}
0x00603fd28964766e9351c0827e2938cf4fb20d   -> {'createTokens', 'mint', 'calculateAndTransferTokens'}


In [ ]:
hasil_label_v2 = dict(hasil_label)  # salin hasil lama sebagai basis
versi_aktif, mulai = None, time.time()

for i, nama in enumerate(kosong, 1):
    r = df_ren[df_ren.nama_file == nama].iloc[0]
    try:
        if r.solc != versi_aktif:
            subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
            versi_aktif = r.solc
        hasil_label_v2[nama] = label_fungsi_reentrancy_v2(r.path, str(r.solc))
    except Exception:
        hasil_label_v2[nama] = set()
    if i % 25 == 0:
        total_pulih = sum(1 for n in kosong[:i] if len(hasil_label_v2[n]) > 0)
        print(f'[{i}/{len(kosong)}] {total_pulih} dari yang kosong sekarang terisi ({(time.time()-mulai)/60:.1f} mnt)')

total_pulih = sum(1 for n in kosong if len(hasil_label_v2[n]) > 0)
total_rentan_akhir = sum(len(v) > 0 for v in hasil_label_v2.values())
total_fungsi_akhir = sum(len(v) for v in hasil_label_v2.values())

print(f'\nselesai dalam {(time.time()-mulai)/60:.1f} menit')
print(f'dari 139 yang kosong, {total_pulih} sekarang terisi')
print(f'total kontrak dengan >=1 fungsi rentan: {total_rentan_akhir}/244')
print(f'total fungsi rentan terdeteksi: {total_fungsi_akhir}')

with open(f'{D}/data/processed/label_fungsi_reentrancy.pkl', 'wb') as f:
    pickle.dump(hasil_label_v2, f)
print('tersimpan (menimpa versi lama)')

[25/139] 20 dari yang kosong sekarang terisi (1.5 mnt)
[50/139] 39 dari yang kosong sekarang terisi (6.1 mnt)
[75/139] 54 dari yang kosong sekarang terisi (8.7 mnt)
[100/139] 75 dari yang kosong sekarang terisi (9.3 mnt)
[125/139] 91 dari yang kosong sekarang terisi (10.1 mnt)

selesai dalam 10.4 menit
dari 139 yang kosong, 99 sekarang terisi
total kontrak dengan >=1 fungsi rentan: 204/244
total fungsi rentan terdeteksi: 456
tersimpan (menimpa versi lama)


In [ ]:
with open(f'{D}/data/processed/label_fungsi_reentrancy.pkl', 'rb') as f:
    hasil_label_final = pickle.load(f)

masih_kosong = [n for n, v in hasil_label_final.items() if len(v) == 0]
print(f'{len(masih_kosong)} kontrak tidak dipecah (dipertahankan sebagai graf utuh saja)')
print(f'{244 - len(masih_kosong)} kontrak akan dipecah jadi subgraf per fungsi')

# simpan daftar ini untuk dipakai di tahap pemecahan subgraf nanti
with open(f'{D}/data/processed/kontrak_dipecah_vs_utuh.pkl', 'wb') as f:
    pickle.dump({'dipecah': [n for n in hasil_label_final if n not in masih_kosong],
                'utuh_saja': masih_kosong}, f)
print('tersimpan ke kontrak_dipecah_vs_utuh.pkl')

40 kontrak tidak dipecah (dipertahankan sebagai graf utuh saja)
204 kontrak akan dipecah jadi subgraf per fungsi
tersimpan ke kontrak_dipecah_vs_utuh.pkl


In [ ]:
OUT_CFG = f'{D}/data/processed/v2_cfg'
contoh_file = [f for f in os.listdir(OUT_CFG) if f.startswith('1_reentrancy__')][0]
with open(f'{OUT_CFG}/{contoh_file}') as f:
    contoh = json.load(f)

print('nama file:', contoh_file)
print('kunci yang ada di tiap node:', list(contoh['nodes'][0].keys()))

nama file: 1_reentrancy__0x00391d08b3e68e476a774ada379258264eb74485.json
kunci yang ada di tiap node: ['id', 'tipe', 'n_call_low', 'n_call_high', 'n_internal', 'n_sstore', 'n_sload', 'n_irs']


In [ ]:
def ekstrak_cfg_v2(fp):
    """Sama seperti ekstrak_cfg, tapi menyimpan nama fungsi di tiap node."""
    sl = Slither(fp)
    peta, nodes, fungsi_entry = {}, [], {}
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                gid = len(nodes)
                peta[(c.name, fn.full_name, n.node_id)] = gid
                nodes.append({
                    'id': gid,
                    'fungsi': fn.name,          # <-- tambahan baru
                    'kontrak': c.name,          # <-- tambahan baru, buat jaga-jaga
                    'tipe': str(n.type).replace('NodeType.', ''),
                    'n_call_low': len(n.low_level_calls),
                    'n_call_high': len(n.high_level_calls),
                    'n_internal': len(n.internal_calls),
                    'n_sstore': len(n.state_variables_written),
                    'n_sload': len(n.state_variables_read),
                    'n_irs': len(n.irs),
                })
                if n.node_id == 0:
                    fungsi_entry[(c.name, fn.full_name)] = gid

    edges = []
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                a = peta[(c.name, fn.full_name, n.node_id)]
                for m in n.sons:
                    k = (c.name, fn.full_name, m.node_id)
                    if k in peta:
                        edges.append([a, peta[k], 0])
                for ic in n.internal_calls:
                    target = getattr(ic, 'function', ic)
                    nama = getattr(target, 'full_name', None)
                    if nama and (c.name, nama) in fungsi_entry:
                        edges.append([a, fungsi_entry[(c.name, nama)], 1])

    return {'nodes': nodes, 'edges': edges, 'n_node': len(nodes), 'n_edge': len(edges)}

print('fungsi ekstraksi v2 siap')

fungsi ekstraksi v2 siap


In [ ]:
OUT_CFG_V2 = f'{D}/data/processed/v3_cfg_dengan_fungsi'
os.makedirs(OUT_CFG_V2, exist_ok=True)

# gabungkan sumber: normal dari v2_manifest, reentrancy dari manifest_final
mf_normal = pd.read_csv(MANIFEST)
mf_normal_ok = mf_normal[mf_normal.status == 'OK'].copy()
mf_normal_ok['folder'] = '0_normal'

tugas = []
for _, r in mf_normal_ok.iterrows():
    tugas.append(('0_normal', r.nama_file, r.path, str(r.solc)))
for _, r in df_ren.iterrows():
    tugas.append(('1_reentrancy', r.nama_file, r.path, str(r.solc)))

print(f'{len(tugas)} kontrak akan diekstraksi ulang (dengan atribut fungsi)')

tugas.sort(key=lambda x: x[3])  # urutkan per versi solc biar efisien
sudah = {f[:-5] for f in os.listdir(OUT_CFG_V2) if f.endswith('.json')}

versi_aktif, berhasil, gagal, mulai = None, 0, [], time.time()
for i, (folder, nama, path, ver) in enumerate(tugas, 1):
    kunci = f"{folder}__{nama[:-4]}"
    if kunci in sudah:
        continue
    try:
        if ver != versi_aktif:
            subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
            versi_aktif = ver
        g = ekstrak_cfg_v2(path)
        g.update({'nama_file': nama, 'folder': folder,
                  'label': 0 if folder == '0_normal' else 1, 'split': 'train'})
        with open(f'{OUT_CFG_V2}/{kunci}.json', 'w') as f:
            json.dump(g, f)
        berhasil += 1
        if i % 50 == 0:
            print(f'[{i}/{len(tugas)}] ({(time.time()-mulai)/60:.1f} mnt)')
    except Exception as e:
        gagal.append((nama, str(e)[:80]))

print(f'\nselesai: {berhasil} berhasil, {len(gagal)} gagal, {(time.time()-mulai)/60:.1f} menit')

624 kontrak akan diekstraksi ulang (dengan atribut fungsi)
[50/624] (0.8 mnt)
[100/624] (1.6 mnt)
[150/624] (6.9 mnt)
[200/624] (10.0 mnt)
[250/624] (10.6 mnt)
[300/624] (11.9 mnt)
[350/624] (13.1 mnt)
[400/624] (14.2 mnt)
[450/624] (15.1 mnt)
[500/624] (15.7 mnt)
[550/624] (16.6 mnt)
[600/624] (17.4 mnt)

selesai: 607 berhasil, 17 gagal, 17.8 menit


In [ ]:
def pecah_per_fungsi(g, label_rentan_set=None):
    """g: dict graf lengkap. label_rentan_set: None (semua label 0) atau set nama fungsi rentan."""
    from collections import defaultdict
    kelompok = defaultdict(list)
    for n in g['nodes']:
        kelompok[n['fungsi']].append(n)

    subgraf_list = []
    for nama_fn, nodes_fn in kelompok.items():
        if len(nodes_fn) < 3:
            continue
        id_lama = {n['id'] for n in nodes_fn}
        peta = {old: new for new, old in enumerate(sorted(id_lama))}
        nodes_baru = []
        for n in nodes_fn:
            n2 = dict(n)
            n2['id'] = peta[n['id']]
            nodes_baru.append(n2)
        edges_baru = [[peta[a], peta[b], t] for a, b, t in g['edges']
                     if a in id_lama and b in id_lama]

        if label_rentan_set is None:
            label = 0
        else:
            label = 1 if nama_fn in label_rentan_set else 0

        subgraf_list.append({
            'nodes': nodes_baru, 'edges': edges_baru,
            'n_node': len(nodes_baru), 'n_edge': len(edges_baru),
            'nama_file': f"{g['nama_file']}::{nama_fn}",
            'folder': g['folder'], 'label': label, 'split': 'train',
        })
    return subgraf_list

print('fungsi pemecahan siap')

fungsi pemecahan siap


In [ ]:
with open(f'{D}/data/processed/label_fungsi_reentrancy.pkl', 'rb') as f:
    label_fungsi = pickle.load(f)

OUT_SUBGRAF = f'{D}/data/processed/v3_subgraf_fungsi'
os.makedirs(OUT_SUBGRAF, exist_ok=True)

n_sub_normal, n_sub_ren = 0, 0
for f in sorted(os.listdir(OUT_CFG_V2)):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_CFG_V2}/{f}') as fh:
        g = json.load(fh)

    if g['folder'] == '0_normal':
        subs = pecah_per_fungsi(g, label_rentan_set=None)
        n_sub_normal += len(subs)
    elif g['folder'] == '1_reentrancy':
        set_rentan = label_fungsi.get(g['nama_file'], set())
        if len(set_rentan) == 0:
            continue  # 40 kontrak yang tidak terdeteksi, skip (tetap graf utuh di data lama)
        subs = pecah_per_fungsi(g, label_rentan_set=set_rentan)
        n_sub_ren += len(subs)
    else:
        continue

    for i, sg in enumerate(subs):
        nama_aman = sg['nama_file'].replace('/', '_').replace('.sol', '')
        with open(f'{OUT_SUBGRAF}/{sg["folder"]}__{nama_aman}.json', 'w') as fh:
            json.dump(sg, fh)

print(f'subgraf normal (label 0)    : {n_sub_normal}')
print(f'subgraf reentrancy (label 1 campuran): {n_sub_ren}')
print(f'total subgraf: {n_sub_normal + n_sub_ren}')

subgraf normal (label 0)    : 11860
subgraf reentrancy (label 1 campuran): 16354
total subgraf: 28214


In [ ]:
label0, label1 = 0, 0
for f in os.listdir(OUT_SUBGRAF):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_SUBGRAF}/{f}') as fh:
        g = json.load(fh)
    if g['label'] == 0:
        label0 += 1
    else:
        label1 += 1

print(f'subgraf label 0 (aman) : {label0}')
print(f'subgraf label 1 (rentan): {label1}')
print(f'rasio: {label0/label1:.2f} : 1')

# sebaran ukuran subgraf
ukuran = []
for f in os.listdir(OUT_SUBGRAF):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_SUBGRAF}/{f}') as fh:
        g = json.load(fh)
    ukuran.append(g['n_node'])

ukuran = np.array(ukuran)
print(f'\nsebaran n_node subgraf:')
print(f'  min={ukuran.min()}, median={np.median(ukuran):.0f}, max={ukuran.max()}, mean={ukuran.mean():.1f}')
print(f'  p90={np.percentile(ukuran,90):.0f}, p95={np.percentile(ukuran,95):.0f}')

subgraf label 0 (aman) : 27760
subgraf label 1 (rentan): 454
rasio: 61.15 : 1

sebaran n_node subgraf:
  min=3, median=6, max=912, mean=12.4
  p90=23, p95=36


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
D = '/content/drive/MyDrive/PROJECT'
OUT_PYG = f'{D}/data/processed/pyg'
print(os.listdir(OUT_PYG))

Mounted at /content/drive
['cfg_prune_0.02.pt', 'cfg.pt', 'v2_cfg_prune.pt', 'v2_cfg_prune_pos.pt']


In [ ]:
import json, math
import torch
from torch_geometric.data import Data

TIPE = ['EXPRESSION','ENTRYPOINT','RETURN','VARIABLE','IF','ENDIF',
        'OTHER_ENTRYPOINT','STARTLOOP','ENDLOOP','IFLOOP','ASSEMBLY',
        'ENDASSEMBLY','BREAK','CONTINUE','CATCH','TRY','THROW']
TIPE_IDX = {t: i for i, t in enumerate(TIPE)}
DIM = len(TIPE) + 5

def vektor_node(n):
    v = [0.0] * DIM
    idx = TIPE_IDX.get(n['tipe'])
    if idx is not None:
        v[idx] = 1.0
    o = len(TIPE)
    v[o+0] = math.log1p(n['n_call_low'])
    v[o+1] = math.log1p(n['n_call_high'])
    v[o+2] = math.log1p(n['n_sstore'])
    v[o+3] = math.log1p(n['n_sload'])
    v[o+4] = math.log1p(n['n_irs'])
    return v

def ke_pyg(g):
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    x = torch.tensor([vektor_node(n) for n in nodes], dtype=torch.float)
    if g['edges']:
        ei = torch.tensor([[e[0] for e in g['edges']], [e[1] for e in g['edges']]], dtype=torch.long)
        et = torch.tensor([e[2] for e in g['edges']], dtype=torch.long)
    else:
        ei = torch.empty((2,0), dtype=torch.long)
        et = torch.empty((0,), dtype=torch.long)
    return Data(x=x, edge_index=ei, edge_type=et,
                y=torch.tensor([g['label']], dtype=torch.long),
                nama=g['nama_file'], split=g.get('split','train'), folder=g['folder'])

OUT_CFG_V2 = f'{D}/data/processed/v3_cfg_dengan_fungsi'
OUT_PRUNE_SUB = f'{D}/data/processed/v3_subgraf_pruned'
OUT_PRUNE_BM_LAMA = f'{D}/data/processed/v2_cfg_prune_0.02'

print('cek folder sumber ada semua:')
for p in [OUT_CFG_V2, OUT_PRUNE_SUB, OUT_PRUNE_BM_LAMA]:
    print(f'  {p}: {"ADA, " + str(len(os.listdir(p))) + " file" if os.path.exists(p) else "TIDAK ADA"}')

cek folder sumber ada semua:
  /content/drive/MyDrive/PROJECT/data/processed/v3_cfg_dengan_fungsi: ADA, 607 file
  /content/drive/MyDrive/PROJECT/data/processed/v3_subgraf_pruned: TIDAK ADA
  /content/drive/MyDrive/PROJECT/data/processed/v2_cfg_prune_0.02: ADA, 627 file


In [ ]:
for nama in ['v3_subgraf_fungsi', 'v3_subgraf_final', 'label_fungsi_reentrancy.pkl', 'kontrak_dipecah_vs_utuh.pkl']:
    p = f'{D}/data/processed/{nama}'
    if os.path.isdir(p):
        print(f'{nama}: ADA (folder), {len(os.listdir(p))} file')
    elif os.path.isfile(p):
        print(f'{nama}: ADA (file)')
    else:
        print(f'{nama}: TIDAK ADA')

v3_subgraf_fungsi: ADA (folder), 20586 file
v3_subgraf_final: TIDAK ADA
label_fungsi_reentrancy.pkl: ADA (file)
kontrak_dipecah_vs_utuh.pkl: ADA (file)


# Penggabungan Final: Sampling, Pruning, dan Konversi ke PyTorch Geometric

In [ ]:
import os
D = '/content/drive/MyDrive/PROJECT'

print('=== cek semua file/folder penting ===')
cek = {
    'v3_augmented.pt': f'{D}/data/processed/pyg/v3_augmented.pt',
    'v3_subgraf_fungsi (folder)': f'{D}/data/processed/v3_subgraf_fungsi',
    'v3_subgraf_final (folder)': f'{D}/data/processed/v3_subgraf_final',
    'v3_subgraf_pruned (folder)': f'{D}/data/processed/v3_subgraf_pruned',
    'v3_cfg_dengan_fungsi (folder)': f'{D}/data/processed/v3_cfg_dengan_fungsi',
}

for nama, path in cek.items():
    if os.path.isdir(path):
        print(f'{nama}: ADA (folder), {len(os.listdir(path))} file')
    elif os.path.isfile(path):
        print(f'{nama}: ADA (file)')
    else:
        print(f'{nama}: TIDAK ADA')

=== cek semua file/folder penting ===
v3_augmented.pt: TIDAK ADA
v3_subgraf_fungsi (folder): ADA (folder), 20586 file
v3_subgraf_final (folder): TIDAK ADA
v3_subgraf_pruned (folder): TIDAK ADA
v3_cfg_dengan_fungsi (folder): ADA (folder), 607 file


In [ ]:
import shutil
if os.path.exists('/content/v3_subgraf_fungsi'):
    shutil.rmtree('/content/v3_subgraf_fungsi')
    print('folder tujuan dibersihkan, mulai ulang dari nol')

folder tujuan dibersihkan, mulai ulang dari nol


In [ ]:
import subprocess
proses = subprocess.Popen(['cp', '-r', f'{D}/data/processed/v3_subgraf_fungsi', '/content/v3_subgraf_fungsi'])
print('proses copy dimulai (sekali saja), PID:', proses.pid)

proses copy dimulai (sekali saja), PID: 18731


In [ ]:
import os
if os.path.exists('/content/v3_subgraf_fungsi'):
    n = len(os.listdir('/content/v3_subgraf_fungsi'))
    print(f'{n} file tersalin sejauh ini')
    if proses.poll() is not None:
        print('SELESAI, kode keluar:', proses.poll())
    else:
        print('masih berjalan...')
else:
    print('folder belum terbentuk')

20587 file tersalin sejauh ini
SELESAI, kode keluar: 0


In [ ]:
import random, time

OUT_SUBGRAF = '/content/v3_subgraf_fungsi'  # local disk, bukan Drive lagi
mulai = time.time()

file_label1, file_label0 = [], []
for f in sorted(os.listdir(OUT_SUBGRAF)):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_SUBGRAF}/{f}') as fh:
        g = json.load(fh)
    (file_label1 if g['label'] == 1 else file_label0).append(f)

print(f'selesai dalam {time.time()-mulai:.1f} detik')
print(f'label 1: {len(file_label1)}, label 0: {len(file_label0)}')

selesai dalam 3.0 detik
label 1: 264, label 0: 20322


In [ ]:
import pickle
with open(f'{D}/data/processed/label_fungsi_reentrancy.pkl', 'rb') as f:
    label_final = pickle.load(f)

total_kontrak_rentan = sum(1 for v in label_final.values() if len(v) > 0)
total_fungsi_rentan = sum(len(v) for v in label_final.values())
print(f'label final (pkl): {total_kontrak_rentan} kontrak, {total_fungsi_rentan} fungsi rentan')

label final (pkl): 204 kontrak, 456 fungsi rentan


In [ ]:
def pecah_per_fungsi(g, label_rentan_set=None):
    from collections import defaultdict
    kelompok = defaultdict(list)
    for n in g['nodes']:
        kelompok[n['fungsi']].append(n)

    subgraf_list = []
    for nama_fn, nodes_fn in kelompok.items():
        if len(nodes_fn) < 3:
            continue
        id_lama = {n['id'] for n in nodes_fn}
        peta = {old: new for new, old in enumerate(sorted(id_lama))}
        nodes_baru = []
        for n in nodes_fn:
            n2 = dict(n)
            n2['id'] = peta[n['id']]
            nodes_baru.append(n2)
        edges_baru = [[peta[a], peta[b], t] for a, b, t in g['edges']
                     if a in id_lama and b in id_lama]

        label = 0 if label_rentan_set is None else (1 if nama_fn in label_rentan_set else 0)

        subgraf_list.append({
            'nodes': nodes_baru, 'edges': edges_baru,
            'n_node': len(nodes_baru), 'n_edge': len(edges_baru),
            'nama_file': f"{g['nama_file']}::{nama_fn}",
            'folder': g['folder'], 'label': label, 'split': 'train',
        })
    return subgraf_list

print('fungsi pemecahan siap')

fungsi pemecahan siap


In [ ]:
import shutil, time

if not os.path.exists('/content/v3_cfg_dengan_fungsi'):
    shutil.copytree(f'{D}/data/processed/v3_cfg_dengan_fungsi', '/content/v3_cfg_dengan_fungsi')
print(len(os.listdir('/content/v3_cfg_dengan_fungsi')), 'file CFG tersedia di local')

OUT_CFG_V2 = '/content/v3_cfg_dengan_fungsi'
mulai = time.time()
n_sub_normal, n_sub_ren = 0, 0
subgraf_normal, subgraf_ren = [], []

for f in sorted(os.listdir(OUT_CFG_V2)):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_CFG_V2}/{f}') as fh:
        g = json.load(fh)

    if g['folder'] == '0_normal':
        subs = pecah_per_fungsi(g, label_rentan_set=None)
        subgraf_normal.extend(subs)
        n_sub_normal += len(subs)
    elif g['folder'] == '1_reentrancy':
        set_rentan = label_final.get(g['nama_file'], set())
        if len(set_rentan) == 0:
            continue
        subs = pecah_per_fungsi(g, label_rentan_set=set_rentan)
        subgraf_ren.extend(subs)
        n_sub_ren += len(subs)

print(f'selesai dalam {time.time()-mulai:.1f} detik')
print(f'subgraf normal : {n_sub_normal}')
print(f'subgraf reentrancy (campuran label): {n_sub_ren}')

label0_count = sum(1 for sg in subgraf_normal if sg['label']==0) + sum(1 for sg in subgraf_ren if sg['label']==0)
label1_count = sum(1 for sg in subgraf_ren if sg['label']==1)
print(f'\ntotal label 0: {label0_count}')
print(f'total label 1: {label1_count}')

607 file CFG tersedia di local
selesai dalam 5.3 detik
subgraf normal : 11860
subgraf reentrancy (campuran label): 16354

total label 0: 27760
total label 1: 454


In [ ]:
import random

# gabungkan semua subgraf label 0 (dari normal + dari reentrancy yg label 0)
semua_label0 = [sg for sg in subgraf_normal] + [sg for sg in subgraf_ren if sg['label']==0]
semua_label1 = [sg for sg in subgraf_ren if sg['label']==1]

print(f'label 0 total: {len(semua_label0)}, label 1 total: {len(semua_label1)}')

# filter: buang fungsi trivial
kandidat_label0_filtered = []
for sg in semua_label0:
    n_call = sum(n['n_call_low'] + n['n_call_high'] for n in sg['nodes'])
    n_sstore = sum(n['n_sstore'] for n in sg['nodes'])
    if sg['n_node'] >= 10 or n_call > 0 or n_sstore > 0:
        kandidat_label0_filtered.append(sg)

print(f'kandidat label 0 setelah filter: {len(kandidat_label0_filtered)}')

random.seed(42)
n_ambil = min(len(kandidat_label0_filtered), int(len(semua_label1) * 1.5))
label0_final = random.sample(kandidat_label0_filtered, n_ambil)
print(f'label 0 final: {len(label0_final)}')

terpilih = semua_label1 + label0_final
print(f'total subgraf terpilih: {len(terpilih)}')

label 0 total: 27760, label 1 total: 454
kandidat label 0 setelah filter: 17735
label 0 final: 681
total subgraf terpilih: 1135


In [ ]:
from collections import Counter
import math, sys
sys.setrecursionlimit(20000)

def hitung_idf(cfg_dir):
    df_tipe = Counter()
    n_dok = 0
    for f in os.listdir(cfg_dir):
        if not f.endswith('.json'):
            continue
        n_dok += 1
        with open(f'{cfg_dir}/{f}') as fh:
            tipe_ada = {n['tipe'] for n in json.load(fh)['nodes']}
        for t in tipe_ada:
            df_tipe[t] += 1
    return {t: math.log(n_dok / d) for t, d in df_tipe.items()}, n_dok

def wajib_simpan(n):
    if n['n_call_low'] + n['n_call_high'] > 0: return True
    if n['n_sstore'] > 0: return True
    if n['tipe'] in ('IF', 'IFLOOP'): return True
    if n['tipe'] in ('ENTRYPOINT', 'OTHER_ENTRYPOINT'): return True
    if n['n_internal'] > 0: return True
    return False

def prune_graf(g, idf, ambang):
    tf = Counter(n['tipe'] for n in g['nodes'])
    total = max(len(g['nodes']), 1)
    simpan = set()
    for n in g['nodes']:
        if wajib_simpan(n):
            simpan.add(n['id']); continue
        skor = (tf[n['tipe']] / total) * idf.get(n['tipe'], 0.0)
        if skor >= ambang:
            simpan.add(n['id'])

    keluar = {}
    for a, b, t in g['edges']:
        keluar.setdefault(a, []).append((b, t))

    def telusuri(mulai, tipe_sisi, terlihat):
        hasil = []
        for b, t in keluar.get(mulai, []):
            if b in terlihat: continue
            terlihat.add(b)
            if b in simpan: hasil.append((b, tipe_sisi))
            else: hasil.extend(telusuri(b, tipe_sisi, terlihat))
        return hasil

    edges_baru = set()
    for a, b, t in g['edges']:
        if a not in simpan: continue
        if b in simpan: edges_baru.add((a, b, t))
        else:
            for c, tt in telusuri(b, t, {a, b}):
                edges_baru.add((a, c, tt))

    urut = sorted(simpan)
    peta = {lama: baru for baru, lama in enumerate(urut)}
    asal = {n['id']: n for n in g['nodes']}
    nodes_baru = [dict(asal[lama], id=peta[lama]) for lama in urut]

    hasil = dict(g)
    hasil['nodes'] = nodes_baru
    hasil['edges'] = [[peta[a], peta[b], t] for a, b, t in sorted(edges_baru)]
    hasil['n_node'] = len(nodes_baru)
    hasil['n_edge'] = len(hasil['edges'])
    return hasil

print('fungsi pruning siap')

fungsi pruning siap


In [ ]:
idf, n_dok = hitung_idf(OUT_CFG_V2)
print(f'IDF dihitung dari {n_dok} graf')

AMBANG = 0.02
subgraf_pruned = [prune_graf(sg, idf, AMBANG) for sg in terpilih]
print(f'{len(subgraf_pruned)} subgraf selesai dipruning')

IDF dihitung dari 607 graf
1135 subgraf selesai dipruning


In [ ]:
TIPE = ['EXPRESSION','ENTRYPOINT','RETURN','VARIABLE','IF','ENDIF',
        'OTHER_ENTRYPOINT','STARTLOOP','ENDLOOP','IFLOOP','ASSEMBLY',
        'ENDASSEMBLY','BREAK','CONTINUE','CATCH','TRY','THROW']
TIPE_IDX = {t: i for i, t in enumerate(TIPE)}
DIM = len(TIPE) + 5

def vektor_node(n):
    v = [0.0] * DIM
    idx = TIPE_IDX.get(n['tipe'])
    if idx is not None: v[idx] = 1.0
    o = len(TIPE)
    v[o+0] = math.log1p(n['n_call_low'])
    v[o+1] = math.log1p(n['n_call_high'])
    v[o+2] = math.log1p(n['n_sstore'])
    v[o+3] = math.log1p(n['n_sload'])
    v[o+4] = math.log1p(n['n_irs'])
    return v

def ke_pyg(g):
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    x = torch.tensor([vektor_node(n) for n in nodes], dtype=torch.float)
    if g['edges']:
        ei = torch.tensor([[e[0] for e in g['edges']], [e[1] for e in g['edges']]], dtype=torch.long)
        et = torch.tensor([e[2] for e in g['edges']], dtype=torch.long)
    else:
        ei = torch.empty((2,0), dtype=torch.long)
        et = torch.empty((0,), dtype=torch.long)
    return Data(x=x, edge_index=ei, edge_type=et,
                y=torch.tensor([g['label']], dtype=torch.long),
                nama=g['nama_file'], split=g.get('split','train'), folder=g['folder'])

print('fungsi fitur siap, DIM =', DIM)

fungsi fitur siap, DIM = 22


In [ ]:
train_gab = []

# bagian 1: kontrak utuh dari v3_cfg_dengan_fungsi (local, yang baru kita pakai)
for f in sorted(os.listdir(OUT_CFG_V2)):
    if not f.endswith('.json'): continue
    with open(f'{OUT_CFG_V2}/{f}') as fh:
        g = json.load(fh)
    d = ke_pyg(g)
    if d.num_nodes > 0 and d.edge_index.size(1) > 0:
        train_gab.append(d)

print(f'kontrak utuh: {len(train_gab)} graf')

# bagian 2: subgraf fungsi yang sudah dipruning
n_sebelum = len(train_gab)
for g in subgraf_pruned:
    d = ke_pyg(g)
    if d.num_nodes > 0 and d.edge_index.size(1) > 0:
        train_gab.append(d)

print(f'subgraf ditambahkan: {len(train_gab) - n_sebelum}')
print(f'total data latih: {len(train_gab)}')

import collections
print('label gabungan:', collections.Counter(int(d.y) for d in train_gab))

kontrak utuh: 607 graf
subgraf ditambahkan: 1105
total data latih: 1712
label gabungan: Counter({0: 1014, 1: 698})


In [ ]:
# benchmark dari v2_cfg_prune_0.02 (sudah ada di Drive, kecil, tidak perlu copy)
OUT_PRUNE_BM = f'{D}/data/processed/v2_cfg_prune_0.02'

bench_gab = []
for f in sorted(os.listdir(OUT_PRUNE_BM)):
    if f.startswith('3_expert_benchmark') and f.endswith('.json'):
        with open(f'{OUT_PRUNE_BM}/{f}') as fh:
            g = json.load(fh)
        g['split'] = 'benchmark'
        d = ke_pyg(g)
        if d.num_nodes > 0 and d.edge_index.size(1) > 0:
            bench_gab.append(d)

print(f'benchmark: {len(bench_gab)} graf, dimensi: {bench_gab[0].x.shape[1]}')
print('label benchmark:', collections.Counter(int(d.y) for d in bench_gab))

# simpan ke LOCAL DISK dulu (aman, tidak tergantung sync Drive)
torch.save({'train': train_gab, 'benchmark': bench_gab, 'dim': DIM, 'tipe': TIPE},
           '/content/v3_augmented.pt')
print('\ntersimpan sementara di local disk: /content/v3_augmented.pt')
print('ukuran:', os.path.getsize('/content/v3_augmented.pt') / 1024 / 1024, 'MB')

benchmark: 16 graf, dimensi: 22
label benchmark: Counter({0: 8, 1: 8})

tersimpan sementara di local disk: /content/v3_augmented.pt
ukuran: 47.25523853302002 MB


In [ ]:
!cp /content/v3_augmented.pt "{D}/data/processed/pyg/v3_augmented.pt"

# verifikasi LANGSUNG, baca ulang dari Drive untuk pastikan benar-benar tersimpan
import time
time.sleep(3)  # beri jeda sebentar untuk sync
ukuran_drive = os.path.getsize(f'{D}/data/processed/pyg/v3_augmented.pt')
print(f'ukuran di Drive: {ukuran_drive / 1024 / 1024:.2f} MB')
print(f'ukuran di local: {os.path.getsize("/content/v3_augmented.pt") / 1024 / 1024:.2f} MB')
print('cocok?' , ukuran_drive == os.path.getsize('/content/v3_augmented.pt'))

ukuran di Drive: 47.26 MB
ukuran di local: 47.26 MB
cocok? True


In [ ]:
class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('arsitektur siap')

arsitektur siap


In [ ]:
def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    w = torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float)
    return w.to(DEV)

@torch.no_grad()
def evaluasi(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for b in loader:
        b = b.to(DEV)
        logit = model(b)
        p = torch.softmax(logit, dim=1)[:, 1]
        preds += logit.argmax(1).cpu().tolist()
        probs += p.cpu().tolist()
        ys += b.y.cpu().tolist()
    m = {'acc': accuracy_score(ys, preds), 'prec': precision_score(ys, preds, zero_division=0),
         'rec': recall_score(ys, preds, zero_division=0), 'f1': f1_score(ys, preds, zero_division=0)}
    m['delta_pr'] = abs(m['prec'] - m['rec'])
    try: m['auc'] = roc_auc_score(ys, probs)
    except Exception: m['auc'] = float('nan')
    return m

def latih_satu_fold(tr, va, ModelClass, dim_input, dim_hidden=512, max_epoch=50, sabar=15,
                    lr=1e-3, batch_size=32, verbose=True):
    model = ModelClass(dim_input=dim_input, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)
    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    for ep in range(1, max_epoch + 1):
        model.train()
        total = 0
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * b.num_graphs
        m_va = evaluasi(model, ld_va)
        if verbose and ep % 10 == 0:
            print(f'    ep{ep:3} loss={total/len(tr):.4f} f1_va={m_va["f1"]:.3f}')
        if m_va['f1'] > f1_terbaik:
            f1_terbaik, bobot_terbaik, mandek = m_va['f1'], copy.deepcopy(model.state_dict()), 0
        else:
            mandek += 1
        if mandek >= sabar:
            if verbose: print(f'    early stopping di epoch {ep}')
            break
    model.load_state_dict(bobot_terbaik)
    return model, evaluasi(model, ld_va)

print('fungsi training siap')

fungsi training siap


In [ ]:
y_aug = np.array([int(d.y) for d in train_gab])
print(f'device: {DEV}')
print(f'train: {len(train_gab)}, dimensi: {DIM}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hasil_aug, model_fold_aug = [], []
mulai = time.time()

for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y_aug)), y_aug), 1):
    print(f'\n=== Fold {k}/5  (train {len(i_tr)}, val {len(i_va)}) ===')
    tr = [train_gab[i] for i in i_tr]
    va = [train_gab[i] for i in i_va]
    model, m = latih_satu_fold(tr, va, MultiScaleGAT_MeanPool, dim_input=DIM)
    m['fold'] = k
    hasil_aug.append(m)
    model_fold_aug.append(model)
    print(f'  F1={m["f1"]:.4f} Acc={m["acc"]:.4f} P={m["prec"]:.4f} R={m["rec"]:.4f} AUC={m["auc"]:.4f}')

hf_aug = pd.DataFrame(hasil_aug)
print(f'\n=== Rata-rata 5 fold ({(time.time()-mulai)/60:.1f} menit) ===')
print(hf_aug[['acc','prec','rec','f1','delta_pr','auc']].agg(['mean','std']).round(4))

os.makedirs(f'{D}/hasil', exist_ok=True)
hf_aug.to_csv('/content/kfold_v3_augmented.csv', index=False)
torch.save({'state_dicts': [m.state_dict() for m in model_fold_aug],
            'hasil': hasil_aug, 'dim_hidden': 512, 'dim_input': DIM},
           '/content/model_v3_augmented.pt')
!cp /content/kfold_v3_augmented.csv "{D}/hasil/kfold_v3_augmented.csv"
!cp /content/model_v3_augmented.pt "{D}/hasil/model_v3_augmented.pt"
print('tersimpan ke local dan Drive')

device: cuda
train: 1712, dimensi: 22

=== Fold 1/5  (train 1369, val 343) ===
    ep 10 loss=0.3911 f1_va=0.796
    ep 20 loss=0.3358 f1_va=0.809
    ep 30 loss=0.2959 f1_va=0.842
    ep 40 loss=0.2641 f1_va=0.845
    ep 50 loss=0.2913 f1_va=0.838
  F1=0.8768 Acc=0.9009 P=0.8897 R=0.8643 AUC=0.9557

=== Fold 2/5  (train 1369, val 343) ===
    ep 10 loss=0.4166 f1_va=0.828
    ep 20 loss=0.3834 f1_va=0.878
    ep 30 loss=0.3475 f1_va=0.848
    early stopping di epoch 33
  F1=0.8824 Acc=0.9067 P=0.9091 R=0.8571 AUC=0.9538

=== Fold 3/5  (train 1370, val 342) ===
    ep 10 loss=0.3705 f1_va=0.821
    ep 20 loss=0.3109 f1_va=0.822
    ep 30 loss=0.2954 f1_va=0.857
    ep 40 loss=0.2838 f1_va=0.861
    ep 50 loss=0.2681 f1_va=0.867
  F1=0.8671 Acc=0.8889 P=0.8435 R=0.8921 AUC=0.9364

=== Fold 4/5  (train 1370, val 342) ===
    ep 10 loss=0.4071 f1_va=0.796
    ep 20 loss=0.3274 f1_va=0.833
    ep 30 loss=0.3085 f1_va=0.830
    early stopping di epoch 36
  F1=0.8632 Acc=0.8860 P=0.8425 R=0.

In [ ]:
ld_bench = DataLoader(bench_gab, batch_size=16)
prob_semua_aug = []
with torch.no_grad():
    for m in model_fold_aug:
        m.eval()
        p = []
        for b in ld_bench:
            b = b.to(DEV)
            p += torch.softmax(m(b), dim=1)[:, 1].cpu().tolist()
        prob_semua_aug.append(p)
prob_semua_aug = np.array(prob_semua_aug)
prob_rata_aug = prob_semua_aug.mean(axis=0)

tabel_aug = pd.DataFrame({
    'nama': [d.nama for d in bench_gab],
    'label': [int(d.y) for d in bench_gab],
    'prob_rentan': prob_rata_aug.round(3),
    'pred': (prob_rata_aug >= 0.5).astype(int),
})
tabel_aug['benar'] = tabel_aug.label == tabel_aug.pred
print(tabel_aug.sort_values(['label','nama']).to_string(index=False))

y_b, pr_b = tabel_aug.label, tabel_aug.pred
print(f'\nAcc={accuracy_score(y_b,pr_b):.4f}  P={precision_score(y_b,pr_b,zero_division=0):.4f}  '
      f'R={recall_score(y_b,pr_b,zero_division=0):.4f}  F1={f1_score(y_b,pr_b,zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y_b,tabel_aug.prob_rentan):.4f}')

print('\n=== Analisis per pasangan (data augmented) ===')
cari_aug = dict(zip(tabel_aug.nama, tabel_aug.prob_rentan))
lolos_aug = 0
for f in sorted(tabel_aug[tabel_aug.label==1].nama):
    r = cari_aug[f]
    a = cari_aug.get(f.replace('.sol','_fixed.sol'))
    ok = r > a
    lolos_aug += ok
    print(f'{f:28} rentan={r:.3f}  aman={a:.3f}  {"OK" if ok else "TERBALIK"}')
print(f'\n{lolos_aug}/8 pasangan diurutkan dengan benar')

                         nama  label  prob_rentan  pred  benar
          etherbank_fixed.sol      0        0.001     0   True
         etherstore_fixed.sol      0        0.018     0   True
modifier_reentrancy_fixed.sol      0        0.000     0   True
         reentrance_fixed.sol      0        0.002     0   True
     reentrancy_dao_fixed.sol      0        0.044     0   True
reentrancy_insecure_fixed.sol      0        0.008     0   True
  reentrancy_simple_fixed.sol      0        0.001     0   True
         simple_dao_fixed.sol      0        0.011     0   True
                etherbank.sol      1        0.001     0  False
               etherstore.sol      1        0.021     0  False
      modifier_reentrancy.sol      1        0.015     0  False
               reentrance.sol      1        0.005     0  False
           reentrancy_dao.sol      1        0.165     0  False
      reentrancy_insecure.sol      1        0.130     0  False
        reentrancy_simple.sol      1        0.001     0

In [ ]:
from sklearn.metrics import f1_score
for ambang in [0.02, 0.05, 0.08, 0.1, 0.15]:
    pred_baru = (tabel_aug.prob_rentan >= ambang).astype(int)
    f1 = f1_score(tabel_aug.label, pred_baru, zero_division=0)
    acc = accuracy_score(tabel_aug.label, pred_baru)
    print(f'ambang={ambang:.2f}  F1={f1:.3f}  Acc={acc:.3f}  prediksi rentan={pred_baru.sum()}/16')

ambang=0.02  F1=0.615  Acc=0.688  prediksi rentan=5/16
ambang=0.05  F1=0.545  Acc=0.688  prediksi rentan=3/16
ambang=0.08  F1=0.545  Acc=0.688  prediksi rentan=3/16
ambang=0.10  F1=0.400  Acc=0.625  prediksi rentan=2/16
ambang=0.15  F1=0.222  Acc=0.562  prediksi rentan=1/16


In [ ]:
# kumpulkan probabilitas dari semua fold validasi (gabungan, bukan benchmark)
semua_prob_val, semua_label_val = [], []
for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y_aug)), y_aug)):
    va = [train_gab[i] for i in i_va]
    ld_va = DataLoader(va, batch_size=32)
    model = model_fold_aug[k]
    model.eval()
    with torch.no_grad():
        for b in ld_va:
            b = b.to(DEV)
            p = torch.softmax(model(b), dim=1)[:, 1].cpu().tolist()
            semua_prob_val += p
            semua_label_val += b.y.cpu().tolist()

semua_prob_val = np.array(semua_prob_val)
semua_label_val = np.array(semua_label_val)

print('cari ambang optimal dari data validasi:')
ambang_terbaik, f1_terbaik = 0.5, 0
for amb in np.arange(0.05, 0.95, 0.05):
    pred = (semua_prob_val >= amb).astype(int)
    f1 = f1_score(semua_label_val, pred, zero_division=0)
    if f1 > f1_terbaik:
        f1_terbaik, ambang_terbaik = f1, amb

print(f'ambang optimal (dari validasi): {ambang_terbaik:.2f}, F1 validasi: {f1_terbaik:.4f}')

cari ambang optimal dari data validasi:
ambang optimal (dari validasi): 0.50, F1 validasi: 0.8797


In [ ]:
pred_bench_final = (tabel_aug.prob_rentan >= ambang_terbaik).astype(int)
f1_final = f1_score(tabel_aug.label, pred_bench_final, zero_division=0)
acc_final = accuracy_score(tabel_aug.label, pred_bench_final)
print(f'\n=== HASIL FINAL BENCHMARK (ambang {ambang_terbaik:.2f} dari validasi) ===')
print(f'F1={f1_final:.4f}  Acc={acc_final:.4f}')
print(f'AUC (tidak terpengaruh ambang): {roc_auc_score(tabel_aug.label, tabel_aug.prob_rentan):.4f}')


=== HASIL FINAL BENCHMARK (ambang 0.50 dari validasi) ===
F1=0.0000  Acc=0.5000
AUC (tidak terpengaruh ambang): 0.7031


In [ ]:
tabel_aug['pred_ambang_optimal'] = pred_bench_final
tabel_aug.to_csv('/content/benchmark_v3_augmented_final.csv', index=False)
!cp /content/benchmark_v3_augmented_final.csv "{D}/hasil/benchmark_v3_augmented_final.csv"
print('tersimpan')

tersimpan


In [ ]:
cek_file = {
    'v3_augmented.pt': f'{D}/data/processed/pyg/v3_augmented.pt',
    'model_v3_augmented.pt': f'{D}/hasil/model_v3_augmented.pt',
    'kfold_v3_augmented.csv': f'{D}/hasil/kfold_v3_augmented.csv',
    'benchmark_v3_augmented_final.csv': f'{D}/hasil/benchmark_v3_augmented_final.csv',
}
for nama, path in cek_file.items():
    if os.path.exists(path):
        print(f'{nama}: ADA ({os.path.getsize(path)/1024/1024:.2f} MB)')
    else:
        print(f'{nama}: TIDAK ADA')

v3_augmented.pt: ADA (47.26 MB)
model_v3_augmented.pt: ADA (12.04 MB)
kfold_v3_augmented.csv: ADA (0.00 MB)
benchmark_v3_augmented_final.csv: ADA (0.00 MB)


# SHAP V3 AUGMENTED

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric

import os, time, pickle, itertools
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import factorial
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

D = '/content/drive/MyDrive/PROJECT'
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV)

class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

paket = torch.load(f'{D}/data/processed/pyg/v3_augmented.pt', weights_only=False)
data_bench = paket['benchmark']
DIM = paket['dim']

simpan = torch.load(f'{D}/hasil/model_v3_augmented.pt', weights_only=False)
models_v3 = []
for sd in simpan['state_dicts']:
    m = MultiScaleGAT_MeanPool(dim_input=DIM, dim_hidden=simpan['dim_hidden']).to(DEV)
    m.load_state_dict(sd)
    m.eval()
    models_v3.append(m)

print(f'benchmark: {len(data_bench)} graf, model: {len(models_v3)} fold, dim: {DIM}')
print('ukuran graf benchmark:', sorted(d.num_nodes for d in data_bench))

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.8 MB/s eta 0:00:00
device: cuda
benchmark: 16 graf, model: 5 fold, dim: 22
ukuran graf benchmark: [6, 6, 7, 7, 7, 7, 7, 7, 7, 9, 9, 10, 10, 10, 10, 14]


In [ ]:
@torch.no_grad()
def nilai_semua_subset(models, data):
    n = data.num_nodes
    daftar = []
    for bits in range(2 ** n):
        mask = torch.tensor([(bits >> i) & 1 for i in range(n)], dtype=torch.float).unsqueeze(1)
        daftar.append(Data(x=data.x * mask, edge_index=data.edge_index))
    loader = DataLoader(daftar, batch_size=512)
    hasil = []
    for b in loader:
        b = b.to(DEV)
        p = np.mean([torch.softmax(m(b), 1)[:, 1].cpu().numpy() for m in models], axis=0)
        hasil.extend(p.tolist())
    return np.array(hasil)

def shapley_dari_cache(v, n):
    nilai = np.zeros(n)
    for i in range(n):
        for bits in range(2 ** n):
            if (bits >> i) & 1:
                continue
            r = bin(bits).count('1')
            bobot = factorial(r) * factorial(n - r - 1) / factorial(n)
            nilai[i] += bobot * (v[bits | (1 << i)] - v[bits])
    return nilai

mulai = time.time()
hasil_shap_v3 = {}
for d in data_bench:
    if d.num_nodes > 15:
        print(f'{d.nama} dilewati (n={d.num_nodes})')
        continue
    v = nilai_semua_subset(models_v3, d)
    hasil_shap_v3[d.nama] = shapley_dari_cache(v, d.num_nodes)
    print(f'{d.nama:32} n={d.num_nodes:2}  selesai ({time.time()-mulai:.0f} dtk)')

print(f'\ntotal: {(time.time()-mulai)/60:.1f} menit')

etherbank_fixed.sol              n= 7  selesai (1 dtk)
etherstore_fixed.sol             n=10  selesai (2 dtk)
modifier_reentrancy_fixed.sol    n= 7  selesai (2 dtk)
reentrance_fixed.sol             n=10  selesai (3 dtk)
reentrancy_dao_fixed.sol         n= 9  selesai (3 dtk)
reentrancy_insecure_fixed.sol    n= 6  selesai (3 dtk)
reentrancy_simple_fixed.sol      n= 7  selesai (3 dtk)
simple_dao_fixed.sol             n= 7  selesai (3 dtk)
etherbank.sol                    n= 7  selesai (3 dtk)
etherstore.sol                   n=10  selesai (4 dtk)
modifier_reentrancy.sol          n=14  selesai (12 dtk)
reentrance.sol                   n=10  selesai (12 dtk)
reentrancy_dao.sol               n= 9  selesai (12 dtk)
reentrancy_insecure.sol          n= 6  selesai (13 dtk)
reentrancy_simple.sol            n= 7  selesai (13 dtk)
simple_dao.sol                   n= 7  selesai (13 dtk)

total: 0.2 menit


In [ ]:
TIPE = ['EXPRESSION','ENTRYPOINT','RETURN','VARIABLE','IF','ENDIF',
        'OTHER_ENTRYPOINT','STARTLOOP','ENDLOOP','IFLOOP','ASSEMBLY',
        'ENDASSEMBLY','BREAK','CONTINUE','CATCH','TRY','THROW']

def node_gt(x):
    return (x[:, 17] > 0) | (x[:, 18] > 0) | (x[:, 19] > 0)

baris, skor_iou = [], []
for d in data_bench:
    if d.nama not in hasil_shap_v3:
        continue
    nilai = hasil_shap_v3[d.nama]
    gt = set(np.where(node_gt(d.x).numpy())[0])
    top2 = set(np.argsort(-nilai)[:2])
    iou = len(top2 & gt) / len(top2 | gt) if gt else None
    pola = 'flat' if np.std(nilai) < 0.001 else 'discriminative'
    top1_gt = int(np.argmax(nilai)) in gt
    baris.append({'kontrak': d.nama, 'std_shap': round(float(np.std(nilai)), 5),
                  'pola': pola, 'top1_gt': top1_gt,
                  'iou_k2': None if iou is None else round(iou, 3)})
    if iou is not None:
        skor_iou.append(iou)

tabel_shap = pd.DataFrame(baris)
print(tabel_shap.to_string(index=False))
print(f'\nrata-rata IoU (k=2) v3: {np.mean(skor_iou):.3f}   (v2 sebelumnya: 0.277)')
print('jumlah pola flat          :', (tabel_shap.pola == 'flat').sum(), 'dari', len(tabel_shap))
print('jumlah pola discriminative:', (tabel_shap.pola == 'discriminative').sum())

tabel_shap.to_csv('/content/shap_v3_ringkasan.csv', index=False)
with open('/content/shap_v3_nilai.pkl', 'wb') as f:
    pickle.dump(hasil_shap_v3, f)
!cp /content/shap_v3_ringkasan.csv "{D}/hasil/shap_v3_ringkasan.csv"
!cp /content/shap_v3_nilai.pkl "{D}/hasil/shap_v3_nilai.pkl"
print('ukuran di Drive:', os.path.getsize(f'{D}/hasil/shap_v3_nilai.pkl'), 'byte')

                      kontrak  std_shap           pola  top1_gt  iou_k2
          etherbank_fixed.sol   0.00010           flat    False   0.000
         etherstore_fixed.sol   0.00957 discriminative    False   0.000
modifier_reentrancy_fixed.sol   0.00013           flat    False   0.250
         reentrance_fixed.sol   0.00039           flat    False   0.000
     reentrancy_dao_fixed.sol   0.00873 discriminative     True   0.400
reentrancy_insecure_fixed.sol   0.06552 discriminative    False   0.333
  reentrancy_simple_fixed.sol   0.00010           flat    False   0.000
         simple_dao_fixed.sol   0.00422 discriminative     True   0.250
                etherbank.sol   0.00013           flat    False   0.000
               etherstore.sol   0.01076 discriminative    False   0.000
      modifier_reentrancy.sol   0.00274 discriminative    False   0.167
               reentrance.sol   0.00182 discriminative    False   0.000
           reentrancy_dao.sol   0.02688 discriminative     True 

In [ ]:
@torch.no_grad()
def v_satu(models, d, mask):
    x = d.x.clone()
    x[torch.tensor(mask) == 0] = 0
    g = Data(x=x, edge_index=d.edge_index,
             batch=torch.zeros(d.num_nodes, dtype=torch.long)).to(DEV)
    return float(np.mean([torch.softmax(m(g), 1)[0, 1].item() for m in models]))

def shapley_lambat(models, d):
    n = d.num_nodes
    nilai = np.zeros(n)
    for i in range(n):
        lain = [j for j in range(n) if j != i]
        for r in range(len(lain) + 1):
            for sub in itertools.combinations(lain, r):
                bobot = factorial(r) * factorial(n - r - 1) / factorial(n)
                m0 = [1 if j in sub else 0 for j in range(n)]
                m1 = list(m0); m1[i] = 1
                nilai[i] += bobot * (v_satu(models, d, m1) - v_satu(models, d, m0))
    return nilai

d_uji = next(d for d in data_bench if d.num_nodes <= 7)
lama = shapley_lambat(models_v3, d_uji)
baru = shapley_dari_cache(nilai_semua_subset(models_v3, d_uji), d_uji.num_nodes)
print(d_uji.nama, 'n =', d_uji.num_nodes)
print('selisih terbesar:', np.abs(lama - baru).max())
print('identik?', np.allclose(lama, baru, atol=1e-5))

etherbank_fixed.sol n = 7
selisih terbesar: 4.6685768471325505e-11
identik? True


In [ ]:
print(TIPE[16], '← index 16, seharusnya tipe node terakhir')
print('index 17-19 seharusnya: n_call_low, n_call_high, n_sstore')

THROW ← index 16, seharusnya tipe node terakhir
index 17-19 seharusnya: n_call_low, n_call_high, n_sstore


# ABLATION STUDY

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric

import os, json, math, time, copy
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, SAGPooling, global_mean_pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

D = '/content/drive/MyDrive/PROJECT'
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV)

class MultiScaleGAT_MeanPool(nn.Module):
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        return global_mean_pool(h3, batch)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('arsitektur utama siap')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
arsitektur utama siap


In [ ]:
paket = torch.load(f'{D}/data/processed/pyg/v3_augmented.pt', weights_only=False)
train_gab = paket['train']
bench_gab = paket['benchmark']
DIM = paket['dim']
y_aug = np.array([int(d.y) for d in train_gab])
print(f'train: {len(train_gab)} graf, label={dict(zip(*np.unique(y_aug, return_counts=True)))}')
print(f'benchmark: {len(bench_gab)} graf, dimensi: {DIM}')

train: 1712 graf, label={np.int64(0): np.int64(1014), np.int64(1): np.int64(698)}
benchmark: 16 graf, dimensi: 22


In [ ]:
def bobot_kelas(dataset):
    y = np.array([int(d.y) for d in dataset])
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    w = torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float)
    return w.to(DEV)

@torch.no_grad()
def evaluasi(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    for b in loader:
        b = b.to(DEV)
        logit = model(b)
        p = torch.softmax(logit, dim=1)[:, 1]
        preds += logit.argmax(1).cpu().tolist()
        probs += p.cpu().tolist()
        ys += b.y.cpu().tolist()
    m = {'acc': accuracy_score(ys, preds), 'prec': precision_score(ys, preds, zero_division=0),
         'rec': recall_score(ys, preds, zero_division=0), 'f1': f1_score(ys, preds, zero_division=0)}
    m['delta_pr'] = abs(m['prec'] - m['rec'])
    try: m['auc'] = roc_auc_score(ys, probs)
    except Exception: m['auc'] = float('nan')
    return m

def latih_satu_fold(tr, va, ModelClass, dim_input, dim_hidden=512, max_epoch=50, sabar=15,
                    lr=1e-3, batch_size=32, verbose=True):
    model = ModelClass(dim_input=dim_input, dim_hidden=dim_hidden).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = torch.nn.CrossEntropyLoss(weight=bobot_kelas(tr))
    ld_tr = DataLoader(tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(va, batch_size=batch_size)
    f1_terbaik, bobot_terbaik, mandek = -1, None, 0
    for ep in range(1, max_epoch + 1):
        model.train()
        total = 0
        for b in ld_tr:
            b = b.to(DEV)
            opt.zero_grad()
            loss = lossf(model(b), b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * b.num_graphs
        m_va = evaluasi(model, ld_va)
        if verbose and ep % 10 == 0:
            print(f'    ep{ep:3} loss={total/len(tr):.4f} f1_va={m_va["f1"]:.3f}')
        if m_va['f1'] > f1_terbaik:
            f1_terbaik, bobot_terbaik, mandek = m_va['f1'], copy.deepcopy(model.state_dict()), 0
        else:
            mandek += 1
        if mandek >= sabar:
            if verbose: print(f'    early stopping di epoch {ep}')
            break
    model.load_state_dict(bobot_terbaik)
    return model, evaluasi(model, ld_va)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('fungsi training dan skf siap')

fungsi training dan skf siap


In [ ]:
from torch_geometric.nn import SAGPooling

class GAT_SingleScale(nn.Module):
    """Satu lapis GAT saja, bukan tiga -- untuk menguji kontribusi pendekatan multi-scale."""
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        return global_mean_pool(h1, batch)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)


class MultiScaleGAT_SAGPool(nn.Module):
    """Tiga lapis GAT, tapi pooling pakai SAGPooling, bukan mean pooling."""
    def __init__(self, dim_input=22, dim_hidden=512, n_kelas=2, dropout=0.5):
        super().__init__()
        h = dim_hidden
        self.gat1 = GATConv(dim_input, h // 8, heads=8, dropout=dropout)
        self.gat2 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.gat3 = GATConv(h, h // 4, heads=4, dropout=dropout)
        self.proj = nn.Linear(dim_input, h)
        self.norm1 = nn.LayerNorm(h)
        self.norm2 = nn.LayerNorm(h)
        self.norm3 = nn.LayerNorm(h)
        self.pool = SAGPooling(h, ratio=0.5)
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_kelas),
        )
        self.dropout = dropout

    def encode(self, x, edge_index, batch):
        h0 = self.proj(x)
        h1 = self.norm1(F.elu(self.gat1(x, edge_index)) + h0)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)
        h3 = self.norm3(F.elu(self.gat3(h2, edge_index)) + h2)
        x_p, ei_p, _, batch_p, _, _ = self.pool(h3, edge_index, batch=batch)
        return global_mean_pool(x_p, batch_p)

    def forward(self, data):
        z = self.encode(data.x, data.edge_index, data.batch)
        return self.classifier(z)

print('dua varian ablation siap')

dua varian ablation siap


In [ ]:
print('train_gab' in dir(), 'bench_gab' in dir(), 'latih_satu_fold' in dir(), 'skf' in dir())

True True True True


In [ ]:
def jalankan_ablation(ModelClass, nama_konfig, dim_input=DIM):
    hasil_k, model_k = [], []
    mulai = time.time()
    for k, (i_tr, i_va) in enumerate(skf.split(np.zeros(len(y_aug)), y_aug), 1):
        tr = [train_gab[i] for i in i_tr]
        va = [train_gab[i] for i in i_va]
        model, m = latih_satu_fold(tr, va, ModelClass, dim_input=dim_input, verbose=False)
        m['fold'] = k
        hasil_k.append(m)
        model_k.append(model)
        print(f'  [{nama_konfig}] fold {k}: F1={m["f1"]:.4f}')
    print(f'{nama_konfig} selesai dalam {(time.time()-mulai)/60:.1f} menit')
    return hasil_k, model_k

print('=== GAT Single-scale ===')
hasil_single, model_single = jalankan_ablation(GAT_SingleScale, 'single-scale')

print('\n=== Multi-scale GAT + SAGPooling ===')
hasil_sagpool, model_sagpool = jalankan_ablation(MultiScaleGAT_SAGPool, 'SAGPooling')

=== GAT Single-scale ===
  [single-scale] fold 1: F1=0.8857
  [single-scale] fold 2: F1=0.9161
  [single-scale] fold 3: F1=0.8763
  [single-scale] fold 4: F1=0.8600
  [single-scale] fold 5: F1=0.8797
single-scale selesai dalam 2.3 menit

=== Multi-scale GAT + SAGPooling ===
  [SAGPooling] fold 1: F1=0.8487
  [SAGPooling] fold 2: F1=0.8425
  [SAGPooling] fold 3: F1=0.8464
  [SAGPooling] fold 4: F1=0.8027
  [SAGPooling] fold 5: F1=0.8814
SAGPooling selesai dalam 6.5 menit


In [ ]:
hf_single = pd.DataFrame(hasil_single)
hf_sagpool = pd.DataFrame(hasil_sagpool)

print('=== Ringkasan CV single-scale ===')
print(hf_single[['acc','prec','rec','f1','auc']].agg(['mean','std']).round(4))
print('\n=== Ringkasan CV SAGPooling ===')
print(hf_sagpool[['acc','prec','rec','f1','auc']].agg(['mean','std']).round(4))

os.makedirs(f'{D}/hasil', exist_ok=True)
hf_single.to_csv('/content/kfold_ablation_single.csv', index=False)
hf_sagpool.to_csv('/content/kfold_ablation_sagpool.csv', index=False)
torch.save({'state_dicts': [m.state_dict() for m in model_single], 'hasil': hasil_single, 'dim_input': DIM},
           '/content/model_ablation_single.pt')
torch.save({'state_dicts': [m.state_dict() for m in model_sagpool], 'hasil': hasil_sagpool, 'dim_input': DIM},
           '/content/model_ablation_sagpool.pt')
!cp /content/kfold_ablation_single.csv "{D}/hasil/kfold_ablation_single.csv"
!cp /content/kfold_ablation_sagpool.csv "{D}/hasil/kfold_ablation_sagpool.csv"
!cp /content/model_ablation_single.pt "{D}/hasil/model_ablation_single.pt"
!cp /content/model_ablation_sagpool.pt "{D}/hasil/model_ablation_sagpool.pt"
print('tersimpan ke Drive')

=== Ringkasan CV single-scale ===
         acc    prec     rec      f1     auc
mean  0.9018  0.8586  0.9112  0.8836  0.9443
std   0.0191  0.0376  0.0218  0.0205  0.0170

=== Ringkasan CV SAGPooling ===
         acc    prec     rec      f1     auc
mean  0.8703  0.8295  0.8625  0.8443  0.9197
std   0.0248  0.0472  0.0469  0.0280  0.0211
tersimpan ke Drive


In [ ]:
def uji_benchmark(model_list, nama):
    ld = DataLoader(bench_gab, batch_size=16)
    prob_semua = []
    with torch.no_grad():
        for m in model_list:
            m.eval()
            p = []
            for b in ld:
                b = b.to(DEV)
                p += torch.softmax(m(b), dim=1)[:, 1].cpu().tolist()
            prob_semua.append(p)
    prob_rata = np.array(prob_semua).mean(axis=0)
    label = np.array([int(d.y) for d in bench_gab])
    pred = (prob_rata >= 0.5).astype(int)
    f1 = f1_score(label, pred, zero_division=0)
    acc = accuracy_score(label, pred)
    auc = roc_auc_score(label, prob_rata)

    nama_file = [d.nama for d in bench_gab]
    lolos = 0
    for f in sorted([n for n, l in zip(nama_file, label) if l == 1]):
        idx_r = nama_file.index(f)
        f_aman = f.replace('.sol', '_fixed.sol')
        if f_aman in nama_file:
            idx_a = nama_file.index(f_aman)
            lolos += prob_rata[idx_r] > prob_rata[idx_a]
    print(f'{nama:14} F1={f1:.4f}  Acc={acc:.4f}  AUC={auc:.4f}  pasangan benar={lolos}/8')
    return {'konfigurasi': nama, 'f1': f1, 'acc': acc, 'auc': auc, 'pasangan_benar': lolos}

hasil_ringkasan = []
hasil_ringkasan.append(uji_benchmark(model_single, 'single-scale'))
hasil_ringkasan.append(uji_benchmark(model_sagpool, 'SAGPooling'))

single-scale   F1=0.0000  Acc=0.5000  AUC=0.5625  pasangan benar=6/8
SAGPooling     F1=0.0000  Acc=0.5000  AUC=0.3750  pasangan benar=1/8


In [ ]:
tabel_ablation = pd.DataFrame([
    {'konfigurasi': 'SAGPooling (data v2)',        'f1_cv': 0.8612, 'f1_bench': 0.2222, 'auc_bench': 0.5938, 'pasangan_benar': '4/8'},
    {'konfigurasi': 'Mean pooling (data v2)',       'f1_cv': 0.8930, 'f1_bench': 0.3636, 'auc_bench': 0.5625, 'pasangan_benar': '4/8'},
    {'konfigurasi': 'Mean pooling + fitur posisi',  'f1_cv': 0.8850, 'f1_bench': 0.2222, 'auc_bench': 0.5938, 'pasangan_benar': '4/8'},
    {'konfigurasi': 'Mean pooling (data v3, MODEL UTAMA)', 'f1_cv': 0.8798, 'f1_bench': 0.0000, 'auc_bench': 0.7031, 'pasangan_benar': '6/8'},
    {'konfigurasi': 'Single-scale GAT (data v3)',   'f1_cv': 0.8836, 'f1_bench': 0.0000, 'auc_bench': 0.5625, 'pasangan_benar': '6/8'},
    {'konfigurasi': 'SAGPooling (data v3)',         'f1_cv': 0.8443, 'f1_bench': 0.0000, 'auc_bench': 0.3750, 'pasangan_benar': '1/8'},
])
print(tabel_ablation.to_string(index=False))

tabel_ablation.to_csv('/content/tabel_ablation_lengkap.csv', index=False)
!cp /content/tabel_ablation_lengkap.csv "{D}/hasil/tabel_ablation_lengkap.csv"
print('\ntersimpan ke Drive')

                        konfigurasi  f1_cv  f1_bench  auc_bench pasangan_benar
               SAGPooling (data v2) 0.8612    0.2222     0.5938            4/8
             Mean pooling (data v2) 0.8930    0.3636     0.5625            4/8
        Mean pooling + fitur posisi 0.8850    0.2222     0.5938            4/8
Mean pooling (data v3, MODEL UTAMA) 0.8798    0.0000     0.7031            6/8
         Single-scale GAT (data v3) 0.8836    0.0000     0.5625            6/8
               SAGPooling (data v3) 0.8443    0.0000     0.3750            1/8

tersimpan ke Drive


In [ ]:
import os

D = '/content/drive/MyDrive/PROJECT'

cek_file = {
    # tahap augmentasi (v3)
    'v3_augmented.pt (dataset)': f'{D}/data/processed/pyg/v3_augmented.pt',
    'model_v3_augmented.pt (model utama)': f'{D}/hasil/model_v3_augmented.pt',
    'kfold_v3_augmented.csv (hasil CV)': f'{D}/hasil/kfold_v3_augmented.csv',
    'benchmark_v3_augmented_final.csv (hasil benchmark)': f'{D}/hasil/benchmark_v3_augmented_final.csv',
    # tahap SHAP v3
    'shap_v3_ringkasan.csv': f'{D}/hasil/shap_v3_ringkasan.csv',
    'shap_v3_nilai.pkl': f'{D}/hasil/shap_v3_nilai.pkl',
    # tahap ablation
    'kfold_ablation_single.csv': f'{D}/hasil/kfold_ablation_single.csv',
    'kfold_ablation_sagpool.csv': f'{D}/hasil/kfold_ablation_sagpool.csv',
    'model_ablation_single.pt': f'{D}/hasil/model_ablation_single.pt',
    'model_ablation_sagpool.pt': f'{D}/hasil/model_ablation_sagpool.pt',
    'tabel_ablation_lengkap.csv': f'{D}/hasil/tabel_ablation_lengkap.csv',
}

print('=== Verifikasi semua hasil kerja hari ini ===\n')
semua_aman = True
for nama, path in cek_file.items():
    if os.path.exists(path):
        ukuran = os.path.getsize(path)
        satuan = f'{ukuran/1024/1024:.2f} MB' if ukuran > 1024*1024 else f'{ukuran/1024:.2f} KB'
        print(f'[ADA]       {nama:42} {satuan}')
    else:
        print(f'[TIDAK ADA] {nama}')
        semua_aman = False

print()
if semua_aman:
    print('Semua file hari ini sudah tersimpan di Drive. Aman untuk disconnect.')
else:
    print('Ada file yang belum tersimpan, cek yang ditandai TIDAK ADA di atas sebelum disconnect.')

=== Verifikasi semua hasil kerja hari ini ===

[ADA]       v3_augmented.pt (dataset)                  47.26 MB
[ADA]       model_v3_augmented.pt (model utama)        12.04 MB
[ADA]       kfold_v3_augmented.csv (hasil CV)          0.59 KB
[ADA]       benchmark_v3_augmented_final.csv (hasil benchmark) 0.66 KB
[ADA]       shap_v3_ringkasan.csv                      0.83 KB
[ADA]       shap_v3_nilai.pkl                          1.99 KB
[ADA]       kfold_ablation_single.csv                  0.57 KB
[ADA]       kfold_ablation_sagpool.csv                 0.60 KB
[ADA]       model_ablation_single.pt                   1.93 MB
[ADA]       model_ablation_sagpool.pt                  12.07 MB
[ADA]       tabel_ablation_lengkap.csv                 0.34 KB

Semua file hari ini sudah tersimpan di Drive. Aman untuk disconnect.
